In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cyvcf2
from collections import namedtuple


Variant = namedtuple("Variant", ["chrom", "pos", "ref", "alt", "info"])

def get_segment_lengths(vcf: VCF):
    """
    Extract contig lengths from VCF header (predicted VCF only).
    Returns dict: {chrom: length}
    """
    contig_lengths = {}
    for line in vcf.raw_header.splitlines():
        if line.startswith("##contig="):
            # Example: ##contig=<ID=PV249936.1,length=7555>
            parts = line.strip().lstrip("##contig=<").rstrip(">").split(",")
            kv = dict(p.split("=") for p in parts)
            contig_lengths[kv["ID"]] = int(kv["length"])
    return contig_lengths

def load_vcf_variants(pred_vcf_path, truth_vcf_path, af_cutoff=0.0, edge_exclude_k=0, use_segment_lengths=False):
    """
    Parse VCF and return set of (chrom, pos, ref, alt) Variant tuples.
    Optionally filters by:
        - AF (allele frequency)
        - distance from edges of segment (if segment lengths are available)
    """
    
    ## store minor, major variants for predicted, and truth ((pred_minor, pred_major), (truth_minor, truth_major))
    variants = [(set(), set()), (set(), set())]
    pred_vcf = VCF(pred_vcf_path)
    truth_vcf = VCF(truth_vcf_path)

    segment_lengths = get_segment_lengths(pred_vcf) if use_segment_lengths else {}

    for vcf_i, vcf in enumerate([pred_vcf, truth_vcf]):
        for rec in vcf:
            alts = rec.ALT or []
            for i, alt in enumerate(alts):
                if alt is None:
                    continue

                af = rec.INFO.get("AF")
                if isinstance(af, (list, tuple)):
                    af_val = af[i] if i < len(af) else af[0]
                else:
                    af_val = af if af is not None else 1.0

                if af_val < af_cutoff:
                    continue

                # Edge exclusion if segment length is known
                chrom = rec.CHROM
                pos = rec.POS
                seg_len = segment_lengths.get(chrom)
                print(rec.INFO)
                info = [allele for allele in rec.INFO.['DP4'].split(",")]

                if edge_exclude_k > 0 and seg_len:
                    if pos <= edge_exclude_k or pos >= (seg_len - edge_exclude_k + 1):
                        continue  # within k bases of edge
                
                if af_val <= 0.50:
                    variants[vcf_i][0].add(Variant(chrom, pos, rec.REF, alt, info))
                else:
                    variants[vcf_i][1].add(Variant(chrom, pos, rec.REF, alt, info))
                
    # print(variants[0][0], variants[1][0])
    return variants[0], variants[1]

def print_stats(pred_variants, truth_variants, title='Minor Variant Comparison Summary'):
    """Prints summary statistics of two variants with a title

    Args:
        pred_variants (_type_): _description_
        truth_variants (_type_): _description_

    Returns:
        _type_: _description_
    """
    tp = pred_variants & truth_variants
    fp = pred_variants - truth_variants
    fn = truth_variants - pred_variants

    precision = len(tp) / (len(tp) + len(fp)) if (tp or fp) else 0.0
    recall = len(tp) / (len(tp) + len(fn)) if (tp or fn) else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0.0
    
    print(f"\n📊 {title}")
    print(f"True Positives (TP): {len(tp)}")
    print(f"False Positives (FP): {len(fp)}")
    print(f"False Negatives (FN): {len(fn)}")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"F1 Score:   {f1:.4f}")

    return {
        "TP": tp, "FP": fp, "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

def compare_variant_sets(predicted_vcf, truth_vcf,
                         af_cutoff=0.01,
                         edge_exclude_k=0):
    """
    Compares predicted VCF against ground truth (LoFreq) VCF.
    Filters predicted variants near edges (if segment lengths available).
    """
    (pred_variants_minor, pred_variants_major), (truth_variants_minor, truth_variants_major) = load_vcf_variants(
        predicted_vcf,
        truth_vcf,
        af_cutoff=af_cutoff,
        edge_exclude_k=edge_exclude_k,
        use_segment_lengths=True  # only predicted VCF has contig lengths
    )

    major_statistics = print_stats(pred_variants_major, truth_variants_major, title='Major Variant Comparison Summary')
    minor_statistics = print_stats(pred_variants_minor, truth_variants_minor, title='Minor Variant Comparison Summary')
    
    return major_statistics, minor_statistics

In [40]:
k = 19
lofreq = '../../test_outputs/UW_results/SRR33673663_clean_k19/SRR33673663.vcf'
bronko = '../../test_outputs/UW_results/SRR33673663_clean_k19/iv.vcf'

major, minor = compare_variant_sets(bronko, lofreq, af_cutoff=0.01, edge_exclude_k=k)

AttributeError: 'cyvcf2.cyvcf2.INFO' object has no attribute 'DP4'

In [36]:
minor

{'TP': set(),
 'FP': {Variant(chrom='PV249929.1', pos=742, ref='G', alt='A', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c585e9a70>),
  Variant(chrom='PV249929.1', pos=1394, ref='C', alt='A', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c53e82730>),
  Variant(chrom='PV249929.1', pos=1481, ref='T', alt='A', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c53e80fc0>),
  Variant(chrom='PV249929.1', pos=1549, ref='A', alt='G', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c53e809f0>),
  Variant(chrom='PV249929.1', pos=1697, ref='A', alt='G', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c53e81bf0>),
  Variant(chrom='PV249932.1', pos=368, ref='T', alt='A', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c53e80840>),
  Variant(chrom='PV249934.1', pos=57, ref='G', alt='A', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c53d91200>),
  Variant(chrom='PV249934.1', pos=101, ref='A', alt='G', info=<cyvcf2.cyvcf2.INFO object at 0x7f8c53d902d0>),
  Variant(chrom='PV249934.1', pos=214, ref='G', alt='T', info=<cyvcf2.cyvcf2.INFO object at 0x7f8